# Projek Probabilitas dan Statistika — Kelompok 7 (3 TI A)
**Analisis Statistik Pengaruh Harga Produk, Ongkos Kirim, dan Waktu Pengiriman Terhadap Skor Kepuasan Pelanggan E-Commerce (Studi Kasus: Olist Store)**

Anggota: Gede Abhita (2455301073), Farrell Fathurrahman (2455301067), Alif Fatan Akhlaqi (2455301020)
Sumber data: Olist Brazilian E-Commerce (Kaggle) — https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data

Notebook ini terdiri dari **dua tahap** dalam satu alur:

| Tahap | Isi |
|---|---|
| **Tahap 1 — Audit dan pembersihan data** | memuat data, struktur, kamus data, missing value, duplikasi, nilai tidak wajar, outlier, keputusan pembersihan |
| **Tahap 2 — Analisis** | A. Deskriptif (RM 1) · B. Kenormalan · C. Korelasi (RM 2) · D. Regresi (RM 2) · E. Uji hipotesis (RM 3) · F. Probabilitas · G. Sensitivitas · H. Ringkasan |

**Cara menjalankan:** Runtime → *Run all*, lalu upload `3TIA_07_ECommerceOlist.xlsx` saat diminta. Semua grafik dan tabel tersimpan di folder `hasil/` dan diunduh sebagai `hasil_analisis.zip` di sel terakhir. Hanya memakai numpy, pandas, matplotlib, dan scipy.

## 0. Setup

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)                      # supaya hasil simulasi selalu sama

# Warna: biru & oranye untuk perbandingan dua kelompok, abu untuk netral
BIRU, ORANYE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
INK, MUTED, GRID, NETRAL = '#0b0b0b', '#52514e', '#e5e4df', '#c9c8c2'

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': .8, 'axes.axisbelow': True,
    'axes.edgecolor': '#9b9a94', 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'text.color': INK,
    'axes.titleweight': 'bold', 'axes.titlesize': 11, 'axes.titlelocation': 'left',
    'figure.facecolor': 'white'})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', None)

HASIL = 'hasil'
os.makedirs(HASIL, exist_ok=True)
K = {}                                  # wadah angka kunci untuk laporan

def simpan(fig, nama):
    fig.savefig(f'{HASIL}/{nama}.png', bbox_inches='tight', facecolor='white')
    plt.show()

def fmt_p(p):
    return '< 0,001' if p < 0.001 else f'{p:.4f}'.replace('.', ',')

VAR_X = ['Harga_Produk', 'Ongkos_Kirim', 'Waktu_Pengiriman_Hari']
VAR_NUM = VAR_X + ['Skor_Kepuasan']
print('Siap.')

# TAHAP 1 — Audit dan Pembersihan Data

## 1.1 Memuat data
Jalankan sel ini lalu **upload file `3TIA_07_ECommerceOlist.xlsx`** saat muncul tombol *Choose Files*.
(Kalau dijalankan di komputer lokal, otomatis membaca file di folder yang sama.)

In [ ]:
import os
NAMA_FILE = '3TIA_07_ECommerceOlist.xlsx'

try:
    from google.colab import files
    if not os.path.exists(NAMA_FILE):
        uploaded = files.upload()
        NAMA_FILE = list(uploaded.keys())[0]
except ImportError:
    pass  # bukan di Colab

df_raw = pd.read_excel(NAMA_FILE, sheet_name='Data Utama')
kamus  = pd.read_excel(NAMA_FILE, sheet_name='Kamus Data')

print('File       :', NAMA_FILE)
print('Ukuran data:', df_raw.shape[0], 'baris x', df_raw.shape[1], 'kolom')

## 1.2 Struktur dataset

In [ ]:
df_raw.head(10)

In [ ]:
df_raw.info()

In [ ]:
# Ringkasan tipe data per kolom
pd.DataFrame({'Tipe di Python': df_raw.dtypes.astype(str),
              'Jumlah nilai unik': df_raw.nunique()})

## 1.3 Kamus data
Pengecekan *kejelasan variabel, tipe, dan skala* (syarat dari dosen).

In [ ]:
kamus

## 1.4 Pengecekan missing value

In [ ]:
miss = pd.DataFrame({'Jumlah kosong': df_raw.isna().sum(),
                     'Persen (%)': df_raw.isna().mean() * 100})
miss

In [ ]:
# Tipe numerik harus benar-benar numerik (bukan teks)
for c in VAR_NUM:
    print(f'{c:25s} numerik? ->', pd.api.types.is_numeric_dtype(df_raw[c]))

## 1.5 Duplikasi dan unit observasi
Kolom `ID_Pesanan` ternyata bisa muncul lebih dari sekali. Penyebabnya satu pesanan dapat berisi beberapa item produk.
Jadi yang perlu dicek: apakah ada **baris yang sama persis** (duplikat murni), dan berapa pesanan yang punya lebih dari satu baris.

In [ ]:
n_baris        = len(df_raw)
n_pesanan_unik = df_raw['ID_Pesanan'].nunique()
n_dup_id       = df_raw['ID_Pesanan'].duplicated().sum()
n_dup_penuh    = df_raw.duplicated().sum()

print(f'Jumlah baris (item pesanan)        : {n_baris:,}')
print(f'Jumlah ID_Pesanan unik             : {n_pesanan_unik:,}')
print(f'Baris dengan ID_Pesanan berulang   : {n_dup_id:,}')
print(f'Baris duplikat murni (semua kolom) : {n_dup_penuh:,}')

In [ ]:
# Contoh pesanan yang punya lebih dari satu baris
id_ganda = df_raw[df_raw['ID_Pesanan'].duplicated(keep=False)].sort_values('ID_Pesanan')
print('Jumlah baris yang terlibat:', len(id_ganda))
id_ganda.head(8)

In [ ]:
# Pada pesanan multi-item, apakah Ongkos_Kirim / Waktu / Skor sama untuk semua barisnya?
g = id_ganda.groupby('ID_Pesanan').agg(
    n_item=('ID_Pesanan', 'size'),
    ongkir_sama=('Ongkos_Kirim', 'nunique'),
    waktu_sama=('Waktu_Pengiriman_Hari', 'nunique'),
    skor_sama=('Skor_Kepuasan', 'nunique'))
print('Pesanan multi-item                    :', len(g))
print('  - Waktu pengiriman identik          :', (g.waktu_sama == 1).sum())
print('  - Skor kepuasan identik             :', (g.skor_sama == 1).sum())
print('  - Ongkos kirim identik              :', (g.ongkir_sama == 1).sum())

**Catatan interpretasi:** waktu pengiriman selalu identik pada pesanan multi-item, dan skor ulasan identik pada sebagian besar pesanan (keduanya melekat pada *pesanan*), sedangkan ongkos kirim bervariasi per item. Nilai yang terulang itu membuat baris-baris tersebut tidak sepenuhnya independen. Karena itu **unit observasi dalam projek ini kita tetapkan sebagai item pesanan**, dan hal ini dicantumkan sebagai keterbatasan. Di bagian analisis nanti kita bandingkan hasilnya dengan versi satu baris per pesanan (analisis sensitivitas).

## 1.6 Pengecekan nilai tidak wajar

In [ ]:
# Statistik ringkas untuk melihat rentang nilai
df_raw[VAR_NUM].describe().T

In [ ]:
cek = pd.DataFrame({
    'Harga_Produk <= 0'            : [(df_raw['Harga_Produk'] <= 0).sum()],
    'Ongkos_Kirim < 0'             : [(df_raw['Ongkos_Kirim'] < 0).sum()],
    'Ongkos_Kirim = 0 (gratis)'    : [(df_raw['Ongkos_Kirim'] == 0).sum()],
    'Waktu_Pengiriman < 0'         : [(df_raw['Waktu_Pengiriman_Hari'] < 0).sum()],
    'Waktu_Pengiriman = 0'         : [(df_raw['Waktu_Pengiriman_Hari'] == 0).sum()],
    'Skor di luar 1-5'             : [(~df_raw['Skor_Kepuasan'].isin([1,2,3,4,5])).sum()],
    'Skor tidak bulat'             : [(df_raw['Skor_Kepuasan'] % 1 != 0).sum()],
}).T.rename(columns={0: 'Jumlah baris'})
cek

In [ ]:
# Lihat baris dengan waktu pengiriman = 0 hari
df_raw[df_raw['Waktu_Pengiriman_Hari'] == 0]

In [ ]:
# Distribusi skor kepuasan
dist_skor = df_raw['Skor_Kepuasan'].value_counts().sort_index().to_frame('Frekuensi')
dist_skor['Persen (%)'] = dist_skor['Frekuensi'] / len(df_raw) * 100
dist_skor

## 1.7 Audit outlier (hanya dihitung, belum dihapus)
Metode IQR: nilai di luar Q1 − 1,5·IQR atau Q3 + 1,5·IQR.

In [ ]:
def audit_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    bawah, atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((s < bawah) | (s > atas)).sum()
    return pd.Series({'Q1': q1, 'Q3': q3, 'IQR': iqr, 'Batas bawah': bawah,
                      'Batas atas': atas, 'Jumlah outlier': n_out,
                      'Persen outlier (%)': n_out / len(s) * 100,
                      'Min': s.min(), 'Maks': s.max()})

audit_outlier = df_raw[VAR_NUM[:3]].apply(audit_iqr).T
audit_outlier

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, c in zip(ax, VAR_NUM[:3]):
    a.boxplot(df_raw[c], vert=False, flierprops={'markersize': 2, 'alpha': .4})
    a.set_title(c); a.set_yticks([])
plt.suptitle('Boxplot audit outlier (data mentah)', y=1.02)
plt.tight_layout(); plt.show()

## 1.8 Keputusan pembersihan
Aturan dosen: *hindari mengubah/menghapus data asli tanpa alasan yang jelas.* Keputusan kita:

| Temuan | Keputusan | Alasan |
|---|---|---|
| Tidak ada missing value | Tidak ada yang diubah | Data sudah lengkap |
| ID_Pesanan berulang (multi-item) | **Dipertahankan** | Bukan duplikat murni; tiap baris adalah item berbeda (harga berbeda) |
| Outlier harga, ongkir, waktu | **Dipertahankan** | Nilai nyata (barang mahal, jarak jauh, keterlambatan). Ditangani dengan metode robust di analisis |
| Ongkos kirim = 0 | **Dipertahankan** | Wajar (gratis ongkir) |
| Waktu pengiriman = 0 hari | **Dipertahankan tetapi diberi penanda** | Kurang wajar, jumlahnya sangat sedikit; diuji lewat analisis sensitivitas |

Jadi `df` (data analisis) isinya sama dengan data mentah, ditambah kolom penanda. Tidak ada baris yang dihapus.

In [ ]:
df = df_raw.copy()

# Kolom penanda (flag) — tidak menghapus apa pun
df['flag_item_ganda']   = df['ID_Pesanan'].duplicated(keep=False)
df['flag_waktu_nol']    = df['Waktu_Pengiriman_Hari'] == 0
df['flag_ongkir_nol']   = df['Ongkos_Kirim'] == 0

# Kelompok skor untuk rumusan masalah 3: tinggi (4-5) vs rendah (1-3)
df['Kelompok_Ulasan'] = np.where(df['Skor_Kepuasan'] >= 4, 'Tinggi (4-5)', 'Rendah (1-3)')

# Versi pembanding: satu baris per pesanan (harga & ongkir dijumlah per pesanan;
# waktu & skor diambil dari baris pertama) untuk analisis sensitivitas
df_per_pesanan = (df_raw.groupby('ID_Pesanan', as_index=False)
                  .agg(Harga_Produk=('Harga_Produk', 'sum'),
                       Ongkos_Kirim=('Ongkos_Kirim', 'sum'),
                       Waktu_Pengiriman_Hari=('Waktu_Pengiriman_Hari', 'first'),
                       Skor_Kepuasan=('Skor_Kepuasan', 'first')))

print('df (item pesanan)      :', df.shape)
print('df_per_pesanan         :', df_per_pesanan.shape)
df['Kelompok_Ulasan'].value_counts().to_frame('Jumlah')

## 1.9 Ringkasan audit

In [ ]:
ringkasan = pd.DataFrame({
    'Item': ['Jumlah baris (item pesanan)', 'Jumlah ID_Pesanan unik', 'Jumlah variabel (kolom)',
             'Missing value (total)', 'Baris duplikat murni', 'Baris dengan ID_Pesanan berulang',
             'Waktu pengiriman = 0 hari', 'Ongkos kirim = 0',
             'Skor di luar rentang 1-5'],
    'Nilai': [len(df_raw), df_raw['ID_Pesanan'].nunique(), df_raw.shape[1],
              int(df_raw.isna().sum().sum()), int(df_raw.duplicated().sum()),
              int(df_raw['ID_Pesanan'].duplicated().sum()),
              int(df['flag_waktu_nol'].sum()), int(df['flag_ongkir_nol'].sum()),
              int((~df_raw['Skor_Kepuasan'].isin([1,2,3,4,5])).sum())]})
ringkasan

In [ ]:
ringkasan.to_csv(f'{HASIL}/tabel_ringkasan_audit.csv', index=False)
audit_outlier.to_csv(f'{HASIL}/tabel_audit_outlier.csv')
K['audit'] = dict(zip(ringkasan['Item'], [int(v) for v in ringkasan['Nilai']]))
K['audit_outlier'] = {i: {c: float(audit_outlier.loc[i, c]) for c in audit_outlier.columns} for i in audit_outlier.index}
print('Tahap 1 selesai: tabel audit tersimpan di folder hasil/')

# TAHAP 2 — Analisis Statistika dan Probabilitas

**Peta tahap ini (dikaitkan dengan rumusan masalah):**

| Bagian | Isi | Rumusan masalah |
|---|---|---|
| A | Statistika deskriptif dan visualisasi | RM 1 |
| B | Uji kenormalan | syarat uji inferensial |
| C | Analisis korelasi | RM 2 |
| D | Regresi linear berganda + uji asumsi | RM 2 |
| E | Uji hipotesis komparatif | RM 3 |
| F | Probabilitas: distribusi, peluang bersyarat, Binomial, CLT, interval kepercayaan | pengembangan ke distribusi probabilitas |
| G | Analisis sensitivitas | uji ketahanan hasil |
| H | Ringkasan dan penyimpanan hasil | — |

In [ ]:
# Menyiapkan data untuk tahap analisis (df sudah dibuat di Tahap 1: tanpa baris yang dihapus + kolom penanda)
df_pp = df_per_pesanan.copy()
df_pp['Kelompok_Ulasan'] = np.where(df_pp['Skor_Kepuasan'] >= 4, 'Tinggi (4-5)', 'Rendah (1-3)')
N = len(df)
K['n_item'] = N; K['n_pesanan'] = len(df_pp)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')   # tabel analisis: 4 desimal
print(f'Data utama: {N:,} baris | per pesanan: {len(df_pp):,} baris')

## A. Statistika deskriptif dan visualisasi (RM 1)
Ukuran pemusatan (mean, median, modus), penyebaran (standar deviasi, variansi, rentang, IQR, koefisien variasi), dan bentuk distribusi (skewness, kurtosis).

In [ ]:
def deskriptif(s):
    q1, q3 = s.quantile([.25, .75])
    return pd.Series({
        'N': s.count(), 'Mean': s.mean(), 'Median': s.median(), 'Modus': s.mode().iloc[0],
        'Std Dev': s.std(), 'Variansi': s.var(), 'Min': s.min(), 'Q1': q1, 'Q3': q3,
        'Maks': s.max(), 'IQR': q3 - q1, 'Skewness': s.skew(), 'Kurtosis (excess)': s.kurt(),
        'Koef. Variasi (%)': s.std() / s.mean() * 100})

tabel_desk = df[VAR_NUM].apply(deskriptif).T
tabel_desk.to_csv(f'{HASIL}/tabel_deskriptif.csv')
for v in VAR_NUM:
    K[f'desk_{v}'] = {c: float(tabel_desk.loc[v, c]) for c in tabel_desk.columns}
tabel_desk

In [ ]:
# Distribusi frekuensi skor kepuasan
frek_skor = df['Skor_Kepuasan'].value_counts().sort_index().to_frame('Frekuensi')
frek_skor['Persen (%)'] = frek_skor['Frekuensi'] / N * 100
frek_skor['Persen kumulatif (%)'] = frek_skor['Persen (%)'].cumsum()
frek_skor.to_csv(f'{HASIL}/tabel_frekuensi_skor.csv')
K['frek_skor'] = {int(k): float(v) for k, v in frek_skor['Persen (%)'].items()}
frek_skor

In [ ]:
# Gambar 1: histogram tiap variabel (sumbu x dipotong di persentil 99 agar bentuk utama terlihat)
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
judul = {'Harga_Produk': 'Harga Produk (BRL)', 'Ongkos_Kirim': 'Ongkos Kirim (BRL)',
         'Waktu_Pengiriman_Hari': 'Waktu Pengiriman (hari)'}
for a, v in zip(ax.ravel()[:3], VAR_X):
    s = df[v]; batas = s.quantile(.99)
    bins = np.arange(-0.5, batas + 1.5, 1) if v == 'Waktu_Pengiriman_Hari' else 50   # data hari = bilangan bulat
    a.hist(s[s <= batas], bins=bins, color=BIRU, edgecolor='white', linewidth=.4)
    a.axvline(s.mean(), color=ORANYE, lw=2, label=f'Mean = {s.mean():.2f}')
    a.axvline(s.median(), color=INK, lw=1.6, ls='--', label=f'Median = {s.median():.2f}')
    a.set_title(f'Distribusi {judul[v]}'); a.set_xlabel(judul[v] + ' (dipotong di persentil 99)')
    a.set_ylabel('Frekuensi'); a.legend(frameon=False, fontsize=9)
a = ax.ravel()[3]
a.bar(frek_skor.index, frek_skor['Frekuensi'], color=BIRU, width=.7)
for x, f, p in zip(frek_skor.index, frek_skor['Frekuensi'], frek_skor['Persen (%)']):
    a.text(x, f, f'{p:.1f}%', ha='center', va='bottom', fontsize=9, color=MUTED)
a.set_title('Distribusi Skor Kepuasan'); a.set_xlabel('Skor kepuasan (1-5)'); a.set_ylabel('Frekuensi')
a.set_ylim(0, frek_skor['Frekuensi'].max() * 1.12)
plt.tight_layout(); simpan(fig, 'gambar_01_histogram')

In [ ]:
# Deskriptif per kelompok ulasan (Tinggi 4-5 vs Rendah 1-3)
desk_kel = df.groupby('Kelompok_Ulasan')[VAR_X].agg(['count', 'mean', 'median', 'std'])
desk_kel.to_csv(f'{HASIL}/tabel_deskriptif_kelompok.csv')
desk_kel

**Cara membaca:** bila *mean* lebih besar dari *median* dan *skewness* positif, distribusi miring ke kanan (ada nilai-nilai besar yang menarik rata-rata). Skor kepuasan sebaliknya: miring ke kiri karena mayoritas pelanggan memberi skor 5.

## B. Uji kenormalan
Banyak uji inferensial mengasumsikan kenormalan. Dengan N hampir 100 ribu, uji formal seperti Shapiro-Wilk akan menolak normalitas pada penyimpangan sekecil apa pun, jadi kita lihat **tiga petunjuk sekaligus**: uji formal (pada sampel acak 5.000 data, batas aman Shapiro-Wilk), skewness/kurtosis, dan Q-Q plot.

In [ ]:
sampel = df[VAR_NUM].sample(5000, random_state=42)
norm_rows = []
for v in VAR_NUM:
    w, p = stats.shapiro(sampel[v])
    norm_rows.append({'Variabel': v, 'Shapiro-Wilk W': w, 'p-value': p,
                      'Skewness': df[v].skew(), 'Kurtosis': df[v].kurt(),
                      'Normal (alpha=0,05)?': 'Ya' if p > .05 else 'Tidak'})
tabel_norm = pd.DataFrame(norm_rows).set_index('Variabel')
tabel_norm.to_csv(f'{HASIL}/tabel_normalitas.csv')
tabel_norm

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3.4))
for a, v in zip(ax, VAR_NUM):
    (osm, osr), (sl, ic, r) = stats.probplot(df[v].sample(5000, random_state=42), dist='norm')
    a.scatter(osm, osr, s=5, color=BIRU, alpha=.5)
    a.plot(osm, sl * np.array(osm) + ic, color=ORANYE, lw=2)
    a.set_title(v.replace('_', ' ')); a.set_xlabel('Kuantil teoretis (normal)'); a.set_ylabel('Kuantil sampel')
plt.tight_layout(); simpan(fig, 'gambar_02_qqplot')

**Kesimpulan sementara:** keempat variabel tidak berdistribusi normal (titik Q-Q menyimpang dari garis, dan semua p-value < 0,05). Konsekuensinya: (1) untuk korelasi kita pakai **Spearman** di samping Pearson, (2) untuk perbandingan dua kelompok kita pakai **Welch t-test** *dan* **Mann-Whitney** sebagai pembanding, (3) berkat **Teorema Limit Pusat** dengan N sangat besar, uji berbasis rata-rata tetap layak dipakai, dan (4) untuk regresi kita laporkan juga galat baku **robust (HC3)**.

## C. Analisis korelasi (RM 2)
Pearson mengukur hubungan **linear**, Spearman mengukur hubungan **monoton** berbasis peringkat (cocok untuk data miring/ordinal seperti skor 1-5). Hipotesis tiap pasangan: H0: ρ = 0 vs H1: ρ ≠ 0, α = 0,05.

In [ ]:
pasangan = [('Harga_Produk', 'Skor_Kepuasan'), ('Ongkos_Kirim', 'Skor_Kepuasan'),
            ('Waktu_Pengiriman_Hari', 'Skor_Kepuasan'),
            ('Ongkos_Kirim', 'Waktu_Pengiriman_Hari'), ('Harga_Produk', 'Ongkos_Kirim'),
            ('Harga_Produk', 'Waktu_Pengiriman_Hari')]
def kekuatan(r):
    a = abs(r)
    return 'sangat lemah' if a < .2 else 'lemah' if a < .4 else 'sedang' if a < .6 else 'kuat' if a < .8 else 'sangat kuat'

rows = []
for x, y in pasangan:
    rp, pp_ = stats.pearsonr(df[x], df[y]); rs, ps = stats.spearmanr(df[x], df[y])
    rows.append({'Pasangan': f'{x} - {y}', 'Pearson r': rp, 'p (Pearson)': pp_,
                 'Spearman rho': rs, 'p (Spearman)': ps, 'Kekuatan (Spearman)': kekuatan(rs),
                 'Arah': 'negatif' if rs < 0 else 'positif'})
tabel_korelasi = pd.DataFrame(rows).set_index('Pasangan')
tabel_korelasi.to_csv(f'{HASIL}/tabel_korelasi.csv')
for (x, y), r in zip(pasangan, rows):
    K[f'kor_{x}_{y}'] = {k: (float(v) if not isinstance(v, str) else v) for k, v in r.items() if k != 'Pasangan'}
tabel_korelasi

In [ ]:
# Gambar 3: heatmap korelasi Spearman
cm = LinearSegmentedColormap.from_list('div', [BIRU, '#f3f3f1', ORANYE])
R = df[VAR_NUM].corr(method='spearman')
nama = ['Harga', 'Ongkir', 'Waktu\nKirim', 'Skor']
fig, a = plt.subplots(figsize=(5.6, 4.6))
im = a.imshow(R.values, cmap=cm, vmin=-1, vmax=1)
a.set_xticks(range(4)); a.set_xticklabels(nama); a.set_yticks(range(4)); a.set_yticklabels(nama)
a.grid(False)
for i in range(4):
    for j in range(4):
        a.text(j, i, f'{R.values[i, j]:.2f}', ha='center', va='center', color=INK, fontsize=11)
a.set_title('Matriks korelasi Spearman'); fig.colorbar(im, ax=a, shrink=.8)
for s in a.spines.values(): s.set_visible(False)
plt.tight_layout(); simpan(fig, 'gambar_03_heatmap_korelasi')

In [ ]:
# Gambar 4: rata-rata (dengan selang kepercayaan 95%) tiap variabel pada setiap skor
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, v in zip(ax, VAR_X):
    g = df.groupby('Skor_Kepuasan')[v].agg(['mean', 'sem'])
    a.errorbar(g.index, g['mean'], yerr=1.96 * g['sem'], fmt='o-', color=BIRU, ecolor=BIRU,
               capsize=3, lw=2, ms=7, mec='white')
    a.set_title(f'Rata-rata {judul[v]}'); a.set_xlabel('Skor kepuasan'); a.set_ylabel('Rata-rata')
    a.set_xticks([1, 2, 3, 4, 5])
plt.tight_layout(); simpan(fig, 'gambar_04_rerata_per_skor')

**Cara membaca:** kita menilai arah (negatif = makin besar nilainya, skor makin rendah) dan kekuatan hubungan. Ingat: korelasi bukan sebab-akibat, dan data ini observasional.

## D. Regresi linear berganda (RM 2)
Model: **Skor = β0 + β1·Harga + β2·Ongkir + β3·Waktu + ε**.

Untuk transparansi, OLS kita hitung langsung dengan `numpy` (rumus β = (X'X)⁻¹X'y), lalu dilaporkan: koefisien, galat baku, uji t, selang kepercayaan 95%, koefisien terstandarisasi, R², uji F, dan **galat baku robust HC3** (aman bila ragam galat tidak konstan).

In [ ]:
def ols(data, y, xs):
    X = np.column_stack([np.ones(len(data))] + [data[c].values.astype(float) for c in xs])
    yv = data[y].values.astype(float)
    n, k = X.shape
    XtXi = np.linalg.inv(X.T @ X)
    b = XtXi @ X.T @ yv
    yhat = X @ b; e = yv - yhat
    sse = e @ e; sst = ((yv - yv.mean()) ** 2).sum()
    r2 = 1 - sse / sst; adj = 1 - (1 - r2) * (n - 1) / (n - k)
    s2 = sse / (n - k); se = np.sqrt(s2 * np.diag(XtXi))
    h = np.einsum('ij,jk,ik->i', X, XtXi, X)                       # leverage
    meat = X.T @ (X * ((e / (1 - h)) ** 2)[:, None])               # HC3
    se3 = np.sqrt(np.diag(XtXi @ meat @ XtXi))
    t = b / se; p = 2 * stats.t.sf(np.abs(t), n - k)
    p3 = 2 * stats.t.sf(np.abs(b / se3), n - k)
    tc = stats.t.ppf(.975, n - k)
    F = ((sst - sse) / (k - 1)) / s2; pF = stats.f.sf(F, k - 1, n - k)
    beta_std = np.r_[np.nan, b[1:] * X[:, 1:].std(0, ddof=1) / yv.std(ddof=1)]
    nama = ['Konstanta'] + xs
    tab = pd.DataFrame({'Koefisien (B)': b, 'Galat baku': se, 't': t, 'p-value': p,
                        'CI 95% bawah': b - tc * se, 'CI 95% atas': b + tc * se,
                        'Beta terstandar': beta_std, 'Galat baku HC3': se3, 'p-value HC3': p3}, index=nama)
    return dict(tab=tab, r2=r2, adj=adj, F=F, pF=pF, n=n, k=k, resid=e, fitted=yhat, X=X, rmse=np.sqrt(s2))

M3 = ols(df, 'Skor_Kepuasan', VAR_X)
print(f"N = {M3['n']:,} | R2 = {M3['r2']:.4f} | R2 adj = {M3['adj']:.4f} | "
      f"F = {M3['F']:.2f} | p(F) = {fmt_p(M3['pF'])} | RMSE = {M3['rmse']:.4f}")
M3['tab'].to_csv(f'{HASIL}/tabel_regresi_model3.csv')
M3['tab']

In [ ]:
# Model pembanding yang hanya memakai dua prediktor sesuai RM 2 (ongkir & waktu)
M2 = ols(df, 'Skor_Kepuasan', ['Ongkos_Kirim', 'Waktu_Pengiriman_Hari'])
print(f"Model 2 prediktor : R2 = {M2['r2']:.4f} | R2 adj = {M2['adj']:.4f}")
print(f"Model 3 prediktor : R2 = {M3['r2']:.4f} | R2 adj = {M3['adj']:.4f}")
M2['tab'].to_csv(f'{HASIL}/tabel_regresi_model2.csv')
M2['tab']

In [ ]:
# Uji asumsi regresi
# 1) Multikolinearitas: VIF (aman bila < 5)
vif = {}
for x in VAR_X:
    lain = [c for c in VAR_X if c != x]
    r2x = ols(df, x, lain)['r2']; vif[x] = 1 / (1 - r2x)
tabel_vif = pd.Series(vif, name='VIF').to_frame()

# 2) Homoskedastisitas: uji Breusch-Pagan (LM = n * R2 dari regresi e^2 terhadap X)
aux = ols(pd.DataFrame({'e2': M3['resid'] ** 2, **{c: df[c].values for c in VAR_X}}), 'e2', VAR_X)
lm = M3['n'] * aux['r2']; p_bp = stats.chi2.sf(lm, len(VAR_X))

# 3) Kenormalan residual
skew_res, kurt_res = stats.skew(M3['resid']), stats.kurtosis(M3['resid'])

# 4) Independensi residual: statistik Durbin-Watson (urutan baris data)
dw = np.sum(np.diff(M3['resid']) ** 2) / np.sum(M3['resid'] ** 2)

print(f'Breusch-Pagan : LM = {lm:.2f}, p = {fmt_p(p_bp)}  -> ragam galat {"TIDAK konstan" if p_bp < .05 else "konstan"}')
print(f'Residual      : skewness = {skew_res:.3f}, kurtosis = {kurt_res:.3f}')
print(f'Durbin-Watson : {dw:.3f}  (mendekati 2 = tidak ada autokorelasi berurutan)')
tabel_vif.to_csv(f'{HASIL}/tabel_vif.csv')
K['asumsi'] = {'bp_lm': float(lm), 'bp_p': float(p_bp), 'skew_res': float(skew_res),
               'kurt_res': float(kurt_res), 'dw': float(dw), 'vif': {k: float(v) for k, v in vif.items()}}
tabel_vif

In [ ]:
# Gambar 5: diagnostik residual
idx = np.random.choice(M3['n'], 5000, replace=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(M3['fitted'][idx], M3['resid'][idx], s=5, color=BIRU, alpha=.35)
ax[0].axhline(0, color=ORANYE, lw=2)
ax[0].set_title('Residual vs nilai prediksi'); ax[0].set_xlabel('Nilai prediksi skor'); ax[0].set_ylabel('Residual')
(osm, osr), (sl, ic, r) = stats.probplot(M3['resid'][idx], dist='norm')
ax[1].scatter(osm, osr, s=5, color=BIRU, alpha=.5); ax[1].plot(osm, sl * np.array(osm) + ic, color=ORANYE, lw=2)
ax[1].set_title('Q-Q plot residual'); ax[1].set_xlabel('Kuantil teoretis'); ax[1].set_ylabel('Kuantil residual')
plt.tight_layout(); simpan(fig, 'gambar_05_diagnostik_residual')

In [ ]:
# Ringkasan hasil regresi dalam kalimat (dipakai untuk laporan)
t3 = M3['tab']
for v in VAR_X:
    print(f"{v:24s} B = {t3.loc[v, 'Koefisien (B)']:+.5f} | beta = {t3.loc[v, 'Beta terstandar']:+.3f} | "
          f"p = {fmt_p(t3.loc[v, 'p-value'])} | p(HC3) = {fmt_p(t3.loc[v, 'p-value HC3'])}")
K['reg3'] = {'r2': float(M3['r2']), 'adj': float(M3['adj']), 'F': float(M3['F']), 'pF': float(M3['pF']),
             'tab': {i: {c: float(t3.loc[i, c]) for c in t3.columns} for i in t3.index}}
K['reg2'] = {'r2': float(M2['r2']), 'adj': float(M2['adj']),
             'tab': {i: {c: float(M2['tab'].loc[i, c]) for c in M2['tab'].columns} for i in M2['tab'].index}}

**Cara membaca:** B = perubahan rata-rata skor untuk kenaikan 1 satuan variabel (variabel lain tetap). *Beta terstandar* membandingkan kekuatan pengaruh antar variabel. R² = proporsi variasi skor yang dijelaskan model. Karena skor kepuasan bersifat **ordinal**, regresi linear di sini dipakai sebagai pendekatan; hasilnya kita dukung dengan korelasi Spearman dan galat baku robust.

## E. Uji hipotesis komparatif (RM 3)
Membandingkan rata-rata **waktu pengiriman** antara kelompok ulasan **tinggi (skor 4-5)** dan **rendah (skor 1-3)**.

- H0: μ_tinggi = μ_rendah (tidak ada perbedaan rata-rata waktu pengiriman)
- H1: μ_tinggi ≠ μ_rendah (ada perbedaan)
- α = 0,05, uji dua arah

In [ ]:
tinggi = df.loc[df['Kelompok_Ulasan'] == 'Tinggi (4-5)', 'Waktu_Pengiriman_Hari']
rendah = df.loc[df['Kelompok_Ulasan'] == 'Rendah (1-3)', 'Waktu_Pengiriman_Hari']

tab_grup = pd.DataFrame({'N': [len(tinggi), len(rendah)], 'Mean': [tinggi.mean(), rendah.mean()],
                         'Median': [tinggi.median(), rendah.median()],
                         'Std Dev': [tinggi.std(), rendah.std()]},
                        index=['Tinggi (4-5)', 'Rendah (1-3)'])
tab_grup.to_csv(f'{HASIL}/tabel_kelompok_ulasan.csv')
tab_grup

In [ ]:
# 1) Uji kesamaan ragam (Levene) -> menentukan pakai Welch atau Student
lev_stat, lev_p = stats.levene(tinggi, rendah, center='median')

# 2) Welch t-test (tidak mengasumsikan ragam sama)
n1, n2 = len(rendah), len(tinggi); m1, m2 = rendah.mean(), tinggi.mean(); v1, v2 = rendah.var(), tinggi.var()
se_d = np.sqrt(v1 / n1 + v2 / n2); t_w = (m1 - m2) / se_d
df_w = (v1 / n1 + v2 / n2) ** 2 / ((v1 / n1) ** 2 / (n1 - 1) + (v2 / n2) ** 2 / (n2 - 1))
p_w = 2 * stats.t.sf(abs(t_w), df_w)
beda = m1 - m2; ci_lo, ci_hi = beda - 1.96 * se_d, beda + 1.96 * se_d

# 3) Ukuran efek: Cohen's d
sp = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2)); d_cohen = beda / sp

# 4) Mann-Whitney U (non-parametrik) + probabilitas superioritas
U, p_u = stats.mannwhitneyu(rendah, tinggi, alternative='two-sided'); cles = U / (n1 * n2)

print(f'Levene            : W = {lev_stat:.2f}, p = {fmt_p(lev_p)}  -> ragam {"berbeda" if lev_p < .05 else "sama"}')
print(f'Welch t-test      : t = {t_w:.3f}, df = {df_w:,.1f}, p = {fmt_p(p_w)}')
print(f'Selisih rata-rata : {beda:.3f} hari (Rendah - Tinggi), CI 95% [{ci_lo:.3f}; {ci_hi:.3f}]')
print(f"Cohen's d         : {d_cohen:.3f}")
print(f'Mann-Whitney U    : U = {U:,.0f}, p = {fmt_p(p_u)}, P(waktu_rendah > waktu_tinggi) = {cles:.3f}')
print('Keputusan         :', 'TOLAK H0 (ada perbedaan signifikan)' if p_w < .05 else 'GAGAL TOLAK H0')
K['hipotesis'] = {'mean_tinggi': float(m2), 'mean_rendah': float(m1), 'median_tinggi': float(tinggi.median()),
                  'median_rendah': float(rendah.median()), 'n_tinggi': int(n2), 'n_rendah': int(n1),
                  'sd_tinggi': float(tinggi.std()), 'sd_rendah': float(rendah.std()),
                  'levene_W': float(lev_stat), 'levene_p': float(lev_p), 'welch_t': float(t_w),
                  'welch_df': float(df_w), 'welch_p': float(p_w), 'selisih': float(beda),
                  'ci_lo': float(ci_lo), 'ci_hi': float(ci_hi), 'cohen_d': float(d_cohen),
                  'mw_U': float(U), 'mw_p': float(p_u), 'cles': float(cles)}

In [ ]:
# ANOVA satu arah & Kruskal-Wallis: waktu pengiriman pada kelima level skor
grup = [g['Waktu_Pengiriman_Hari'].values for _, g in df.groupby('Skor_Kepuasan')]
F_a, p_a = stats.f_oneway(*grup)
H_k, p_k = stats.kruskal(*grup)
gm = df['Waktu_Pengiriman_Hari'].mean()
ssb = sum(len(g) * (g.mean() - gm) ** 2 for g in grup); sst_w = ((df['Waktu_Pengiriman_Hari'] - gm) ** 2).sum()
eta2 = ssb / sst_w
print(f'ANOVA        : F = {F_a:.2f}, p = {fmt_p(p_a)}, eta^2 = {eta2:.4f}')
print(f'Kruskal-Wallis: H = {H_k:.2f}, p = {fmt_p(p_k)}')
K['anova'] = {'F': float(F_a), 'p': float(p_a), 'eta2': float(eta2), 'H': float(H_k), 'p_kw': float(p_k)}
rata_skor = df.groupby('Skor_Kepuasan')['Waktu_Pengiriman_Hari'].agg(['count', 'mean', 'median', 'std'])
rata_skor.to_csv(f'{HASIL}/tabel_waktu_per_skor.csv')
rata_skor

In [ ]:
# Gambar 6: boxplot waktu pengiriman per kelompok ulasan
fig, a = plt.subplots(figsize=(6.2, 4.4))
bp = a.boxplot([tinggi, rendah], patch_artist=True,
               showfliers=False, widths=.5, medianprops={'color': INK, 'lw': 2})
for patch, c in zip(bp['boxes'], [BIRU, ORANYE]):
    patch.set_facecolor(c); patch.set_alpha(.85); patch.set_edgecolor('white')
a.set_xticks([1, 2]); a.set_xticklabels(['Tinggi (4-5)', 'Rendah (1-3)'])
a.scatter([1, 2], [tinggi.mean(), rendah.mean()], marker='D', color='white', edgecolor=INK, zorder=5, s=45, label='Rata-rata')
a.set_ylabel('Waktu pengiriman (hari)'); a.set_title('Waktu pengiriman menurut kelompok ulasan')
a.legend(frameon=False); a.text(.99, .01, 'Outlier tidak ditampilkan', transform=a.transAxes, ha='right', fontsize=8, color=MUTED)
plt.tight_layout(); simpan(fig, 'gambar_06_boxplot_kelompok')

**Cara membaca:** bila p-value < 0,05 maka H0 ditolak. Namun dengan N sangat besar, selisih kecil pun bisa signifikan, maka lihat juga **ukuran efek** (Cohen's d: 0,2 kecil, 0,5 sedang, 0,8 besar) dan **selisih nyatanya dalam hari**.

## F. Probabilitas: distribusi, peluang bersyarat, Binomial, CLT, dan interval kepercayaan
Bagian ini menghubungkan data dengan konsep probabilitas yang dipelajari di kelas.

### F.1 Distribusi probabilitas empiris skor kepuasan (fungsi massa peluang)

In [ ]:
pmf = df['Skor_Kepuasan'].value_counts(normalize=True).sort_index()
tab_pmf = pd.DataFrame({'P(X = x)': pmf, 'P(X <= x)': pmf.cumsum(), 'x * P(x)': pmf.index * pmf})
E_x = (pmf.index * pmf).sum(); Var_x = (((pmf.index - E_x) ** 2) * pmf).sum()
p_tinggi = pmf.loc[[4, 5]].sum(); p_rendah = pmf.loc[[1, 2, 3]].sum()
print(f'E(X) = {E_x:.4f} | Var(X) = {Var_x:.4f} | SD = {np.sqrt(Var_x):.4f}')
print(f'P(skor >= 4) = {p_tinggi:.4f} | P(skor <= 3) = {p_rendah:.4f} | P(skor = 5) = {pmf.loc[5]:.4f} | P(skor = 1) = {pmf.loc[1]:.4f}')
K['pmf'] = {'E': float(E_x), 'Var': float(Var_x), 'p_tinggi': float(p_tinggi), 'p_rendah': float(p_rendah),
            'p5': float(pmf.loc[5]), 'p1': float(pmf.loc[1])}
tab_pmf.to_csv(f'{HASIL}/tabel_pmf_skor.csv')
tab_pmf

### F.2 Distribusi Binomial
Misalkan ulasan rendah (skor ≤ 3) dianggap "sukses" dengan peluang **p** (dari data). Dari **n = 20 pelanggan** acak, jumlah ulasan rendah X ~ Binomial(n = 20, p).

In [ ]:
p = p_rendah; n_b = 20
bin_ = stats.binom(n_b, p)
hasil_bin = {'E(X) = n*p': n_b * p, 'SD(X)': np.sqrt(n_b * p * (1 - p)),
             'P(X = 0)': bin_.pmf(0), 'P(X = 3)': bin_.pmf(3),
             'P(X >= 5)': bin_.sf(4), 'P(X <= 2)': bin_.cdf(2)}
tab_bin = pd.Series(hasil_bin, name='Nilai').to_frame(); tab_bin.to_csv(f'{HASIL}/tabel_binomial.csv')
K['binom'] = {k: float(v) for k, v in hasil_bin.items()}; K['binom']['p'] = float(p)
print(f'p (ulasan rendah) = {p:.4f}, n = {n_b}')
tab_bin

In [ ]:
fig, a = plt.subplots(figsize=(6.5, 3.8))
xs = np.arange(0, 13)
a.bar(xs, bin_.pmf(xs), color=BIRU, width=.7)
a.set_xlabel('Jumlah ulasan rendah dari 20 pelanggan (X)'); a.set_ylabel('P(X = x)')
a.set_title(f'Distribusi Binomial (n = 20, p = {p:.3f})'); a.set_xticks(xs)
plt.tight_layout(); simpan(fig, 'gambar_07_binomial')

### F.3 Probabilitas bersyarat: lama pengiriman dan ulasan rendah
Dari data: **P(ulasan rendah | kategori waktu pengiriman)**, serta uji independensi chi-square.

In [ ]:
df['Kategori_Waktu'] = pd.cut(df['Waktu_Pengiriman_Hari'], bins=[-1, 7, 14, 21, np.inf],
                              labels=['0-7 hari', '8-14 hari', '15-21 hari', '> 21 hari'])
ct = pd.crosstab(df['Kategori_Waktu'], df['Kelompok_Ulasan'])
tab_cond = pd.DataFrame({'Jumlah item': ct.sum(axis=1),
                         'Ulasan rendah': ct['Rendah (1-3)'],
                         'P(rendah | kategori)': ct['Rendah (1-3)'] / ct.sum(axis=1),
                         'P(skor 5 | kategori)': df.groupby('Kategori_Waktu')['Skor_Kepuasan'].apply(lambda s: (s == 5).mean())})
tab_cond.to_csv(f'{HASIL}/tabel_peluang_bersyarat.csv')
chi2, p_chi, dof, _ = stats.chi2_contingency(ct)
cramer = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(f'Chi-square independensi: chi2 = {chi2:.2f}, df = {dof}, p = {fmt_p(p_chi)}, Cramer V = {cramer:.3f}')
K['kondisional'] = {str(i): {c: float(tab_cond.loc[i, c]) for c in tab_cond.columns} for i in tab_cond.index}
K['chi2'] = {'chi2': float(chi2), 'df': int(dof), 'p': float(p_chi), 'cramer_v': float(cramer)}
tab_cond

In [ ]:
# Telat vs tidak telat dengan ambang = kuartil 3 waktu pengiriman, risiko relatif, dan teorema Bayes
ambang = df['Waktu_Pengiriman_Hari'].quantile(.75)
telat = df['Waktu_Pengiriman_Hari'] > ambang
rend = df['Kelompok_Ulasan'] == 'Rendah (1-3)'
p_r_telat = rend[telat].mean(); p_r_tidak = rend[~telat].mean()
rr = p_r_telat / p_r_tidak
p_telat_r = telat[rend].mean()                                   # P(telat | rendah)
bayes = p_telat_r * rend.mean() / telat.mean()                   # cek: sama dengan P(rendah | telat)
print(f'Ambang telat (Q3)               : > {ambang:.0f} hari  (P(telat) = {telat.mean():.4f})')
print(f'P(rendah | telat)               : {p_r_telat:.4f}')
print(f'P(rendah | tidak telat)         : {p_r_tidak:.4f}')
print(f'Risiko relatif                  : {rr:.2f}x')
print(f'P(telat | rendah)               : {p_telat_r:.4f}')
print(f'Cek Bayes P(rendah | telat)     : {bayes:.4f}')
K['telat'] = {'ambang': float(ambang), 'p_telat': float(telat.mean()), 'p_r_telat': float(p_r_telat),
              'p_r_tidak': float(p_r_tidak), 'rr': float(rr), 'p_telat_r': float(p_telat_r), 'bayes': float(bayes),
              'p_rendah': float(rend.mean())}

In [ ]:
fig, a = plt.subplots(figsize=(6.8, 4))
a.bar(tab_cond.index.astype(str), tab_cond['P(rendah | kategori)'] * 100, color=BIRU, width=.62)
for i, v in enumerate(tab_cond['P(rendah | kategori)'] * 100):
    a.text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontsize=10, color=MUTED)
a.axhline(rend.mean() * 100, color=ORANYE, lw=2, ls='--', label=f'Peluang keseluruhan = {rend.mean()*100:.1f}%')
a.set_ylabel('Peluang ulasan rendah (%)'); a.set_xlabel('Kategori waktu pengiriman')
a.set_title('Peluang ulasan rendah menurut lama pengiriman'); a.legend(frameon=False)
a.set_ylim(0, tab_cond['P(rendah | kategori)'].max() * 100 * 1.15)
plt.tight_layout(); simpan(fig, 'gambar_08_peluang_bersyarat')

### F.4 Menyesuaikan distribusi teoretis pada waktu pengiriman
Waktu pengiriman bernilai positif dan miring ke kanan. Kita coba **Lognormal** dan **Gamma** (hanya data > 0 hari), lalu bandingkan peluang `P(waktu > 20 hari)` hasil model dengan peluang empiris.

In [ ]:
w = df.loc[df['Waktu_Pengiriman_Hari'] > 0, 'Waktu_Pengiriman_Hari'].astype(float)
sh_ln, loc_ln, sc_ln = stats.lognorm.fit(w, floc=0)
sh_g, loc_g, sc_g = stats.gamma.fit(w, floc=0)
ln, gm_ = stats.lognorm(sh_ln, loc_ln, sc_ln), stats.gamma(sh_g, loc_g, sc_g)

ks_ln = stats.kstest(w, ln.cdf).statistic; ks_g = stats.kstest(w, gm_.cdf).statistic
emp20 = (w > 20).mean()
tab_fit = pd.DataFrame({'Parameter': [f'sigma={sh_ln:.3f}, scale={sc_ln:.2f}', f'a={sh_g:.3f}, scale={sc_g:.2f}'],
                        'Statistik K-S (D)': [ks_ln, ks_g],
                        'P(waktu > 20) model': [ln.sf(20), gm_.sf(20)],
                        'P(waktu > 20) empiris': [emp20, emp20]}, index=['Lognormal', 'Gamma'])
tab_fit.to_csv(f'{HASIL}/tabel_fit_distribusi.csv')
K['fit'] = {'ks_ln': float(ks_ln), 'ks_gamma': float(ks_g), 'p20_ln': float(ln.sf(20)),
            'p20_gamma': float(gm_.sf(20)), 'p20_emp': float(emp20)}
tab_fit

In [ ]:
fig, a = plt.subplots(figsize=(7.4, 4.2))
xx = np.linspace(0.5, 45, 400)
a.hist(w[w <= 45], bins=np.arange(0.5, 45.5, 1), density=True, color=NETRAL, edgecolor='white', linewidth=.4, label='Data (histogram)')
a.plot(xx, ln.pdf(xx), color=BIRU, lw=2.4, label='Lognormal')
a.plot(xx, gm_.pdf(xx), color=ORANYE, lw=2.4, label='Gamma')
a.set_xlabel('Waktu pengiriman (hari)'); a.set_ylabel('Kepadatan'); a.set_title('Kecocokan distribusi pada waktu pengiriman')
a.legend(frameon=False)
plt.tight_layout(); simpan(fig, 'gambar_09_fit_distribusi')

### F.5 Teorema Limit Pusat (simulasi)
Skor kepuasan jelas **tidak normal**. Tetapi bila kita ambil banyak sampel acak berukuran n dan menghitung rata-ratanya, distribusi rata-rata sampel mendekati normal dengan galat baku σ/√n.

In [ ]:
mu, sigma = df['Skor_Kepuasan'].mean(), df['Skor_Kepuasan'].std(ddof=0)
rng = np.random.default_rng(42)
uk = [5, 30, 200]; sim = {}
for n_s in uk:
    sim[n_s] = rng.choice(df['Skor_Kepuasan'].values, size=(5000, n_s)).mean(axis=1)
tab_clt = pd.DataFrame({'SE teoretis (sigma/sqrt n)': [sigma / np.sqrt(n_s) for n_s in uk],
                        'SE simulasi': [sim[n_s].std() for n_s in uk],
                        'Mean rata-rata sampel': [sim[n_s].mean() for n_s in uk]},
                       index=[f'n = {n_s}' for n_s in uk])
tab_clt.to_csv(f'{HASIL}/tabel_clt.csv')
K['clt'] = {str(n_s): {'se_teori': float(sigma / np.sqrt(n_s)), 'se_sim': float(sim[n_s].std())} for n_s in uk}
tab_clt

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6), sharey=False)
for a, n_s in zip(ax, uk):
    a.hist(sim[n_s], bins=40, density=True, color=BIRU, edgecolor='white', linewidth=.4)
    xx = np.linspace(sim[n_s].min(), sim[n_s].max(), 200)
    a.plot(xx, stats.norm.pdf(xx, mu, sigma / np.sqrt(n_s)), color=ORANYE, lw=2.4)
    a.set_title(f'Rata-rata sampel, n = {n_s}'); a.set_xlabel('Rata-rata skor'); a.set_ylabel('Kepadatan')
plt.tight_layout(); simpan(fig, 'gambar_10_clt')

### F.6 Estimasi interval kepercayaan 95%

In [ ]:
rows = []
for v in VAR_NUM:
    s = df[v]; se = s.sem(); tc = stats.t.ppf(.975, len(s) - 1)
    rows.append({'Variabel': v, 'Mean': s.mean(), 'SE': se, 'CI 95% bawah': s.mean() - tc * se, 'CI 95% atas': s.mean() + tc * se})
ph = (df['Skor_Kepuasan'] <= 3).mean(); se_p = np.sqrt(ph * (1 - ph) / N)
rows.append({'Variabel': 'Proporsi ulasan rendah', 'Mean': ph, 'SE': se_p,
             'CI 95% bawah': ph - 1.96 * se_p, 'CI 95% atas': ph + 1.96 * se_p})
tab_ci = pd.DataFrame(rows).set_index('Variabel'); tab_ci.to_csv(f'{HASIL}/tabel_interval_kepercayaan.csv')
K['ci'] = {i: {c: float(tab_ci.loc[i, c]) for c in tab_ci.columns} for i in tab_ci.index}
tab_ci

## G. Analisis sensitivitas
Apakah kesimpulan berubah bila (1) satu baris per pesanan, (2) 13 baris waktu 0 hari dikeluarkan, atau (3) nilai ekstrem (di atas persentil 99) dibuang? Bila arah dan signifikansinya konsisten, kesimpulan dinilai **kokoh**.

In [ ]:
def ringkas(data, label):
    rho_w = stats.spearmanr(data['Waktu_Pengiriman_Hari'], data['Skor_Kepuasan'])[0]
    rho_o = stats.spearmanr(data['Ongkos_Kirim'], data['Skor_Kepuasan'])[0]
    rho_h = stats.spearmanr(data['Harga_Produk'], data['Skor_Kepuasan'])[0]
    m = ols(data, 'Skor_Kepuasan', VAR_X); t = m['tab']
    a = data.loc[data['Kelompok_Ulasan'] == 'Rendah (1-3)', 'Waktu_Pengiriman_Hari']
    b = data.loc[data['Kelompok_Ulasan'] == 'Tinggi (4-5)', 'Waktu_Pengiriman_Hari']
    tw = stats.ttest_ind(a, b, equal_var=False)
    return {'Skenario': label, 'N': len(data), 'rho Waktu': rho_w, 'rho Ongkir': rho_o, 'rho Harga': rho_h,
            'B Waktu': t.loc['Waktu_Pengiriman_Hari', 'Koefisien (B)'],
            'B Ongkir': t.loc['Ongkos_Kirim', 'Koefisien (B)'],
            'B Harga': t.loc['Harga_Produk', 'Koefisien (B)'], 'R2': m['r2'],
            'Selisih waktu (hari)': a.mean() - b.mean(), 'p Welch': tw.pvalue}

p99 = df[VAR_X].quantile(.99)
skenario = [
    (df, 'Data utama (item pesanan)'),
    (df_pp, 'Satu baris per pesanan'),
    (df[df['Waktu_Pengiriman_Hari'] > 0], 'Tanpa waktu kirim 0 hari'),
    (df[(df['Harga_Produk'] <= p99['Harga_Produk']) & (df['Ongkos_Kirim'] <= p99['Ongkos_Kirim'])
        & (df['Waktu_Pengiriman_Hari'] <= p99['Waktu_Pengiriman_Hari'])], 'Tanpa nilai ekstrem (> P99)')]
tab_sens = pd.DataFrame([ringkas(d_, l_) for d_, l_ in skenario]).set_index('Skenario')
tab_sens.to_csv(f'{HASIL}/tabel_sensitivitas.csv')
K['sens'] = {i: {c: float(tab_sens.loc[i, c]) for c in tab_sens.columns} for i in tab_sens.index}
tab_sens

## H. Ringkasan dan penyimpanan hasil

In [ ]:
r = K['hipotesis']; g = K['reg3']['tab']
print('=== RINGKASAN TEMUAN ===')
print(f"RM1  Mean skor = {K['desk_Skor_Kepuasan']['Mean']:.3f}, median = {K['desk_Skor_Kepuasan']['Median']:.0f}; "
      f"{K['pmf']['p_tinggi']*100:.1f}% ulasan bernilai 4-5.")
print(f"RM2  Spearman skor-waktu = {K['kor_Waktu_Pengiriman_Hari_Skor_Kepuasan']['Spearman rho']:+.3f}; "
      f"skor-ongkir = {K['kor_Ongkos_Kirim_Skor_Kepuasan']['Spearman rho']:+.3f}; "
      f"skor-harga = {K['kor_Harga_Produk_Skor_Kepuasan']['Spearman rho']:+.3f}")
print(f"RM2  Regresi: R2 = {K['reg3']['r2']:.4f}; B waktu = {g['Waktu_Pengiriman_Hari']['Koefisien (B)']:+.4f} "
      f"(p = {fmt_p(g['Waktu_Pengiriman_Hari']['p-value'])}); B ongkir = {g['Ongkos_Kirim']['Koefisien (B)']:+.4f} "
      f"(p = {fmt_p(g['Ongkos_Kirim']['p-value'])})")
print(f"RM3  Waktu kirim rata-rata: rendah = {r['mean_rendah']:.2f} hari vs tinggi = {r['mean_tinggi']:.2f} hari; "
      f"selisih {r['selisih']:.2f} hari, p = {fmt_p(r['welch_p'])}, d = {r['cohen_d']:.2f}")
print(f"Prob P(rendah | telat > {K['telat']['ambang']:.0f} hari) = {K['telat']['p_r_telat']:.3f} vs "
      f"tidak telat = {K['telat']['p_r_tidak']:.3f} (RR = {K['telat']['rr']:.2f}x)")

with open(f'{HASIL}/angka_kunci.json', 'w') as f:
    json.dump(K, f, indent=1, default=float)

In [ ]:
# Kemas semua hasil menjadi satu ZIP lalu unduh
import shutil
shutil.make_archive('hasil_analisis', 'zip', HASIL)
try:
    from google.colab import files
    files.download('hasil_analisis.zip')
except ImportError:
    print('Tersimpan: hasil_analisis.zip')